# 03 — YOLOv8 Training
Fine-tune YOLOv8 for brain tumor localization on annotated MRI images.

In [ ]:
# Install ultralytics if not present
# !pip install ultralytics -q
from ultralytics import YOLO
import torch
print(f'PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')

In [ ]:
# Load a pretrained YOLOv8 nano model
model = YOLO('yolov8n.pt')  # Change to yolov8s/m/l for higher accuracy
print('Model loaded.')

In [ ]:
# Train
results = model.train(
    data='../configs/tumor.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    project='../models',
    name='yolov8_brain_tumor',
    save=True,
    val=True,
    verbose=True,
)
print('Training complete!')
print(f'Best weights: {results.save_dir}/weights/best.pt')

In [ ]:
# Validate the best model
best_model = YOLO(f'{results.save_dir}/weights/best.pt')
metrics = best_model.val(data='../configs/tumor.yaml')
print(f'mAP@0.5      : {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95 : {metrics.box.map:.4f}')
print(f'Precision    : {metrics.box.mp:.4f}')
print(f'Recall       : {metrics.box.mr:.4f}')

In [ ]:
# Run inference on a sample image
import cv2
import matplotlib.pyplot as plt

sample_image = '../data/raw/test/glioma/sample.jpg'  # Update path

preds = best_model(sample_image, conf=0.25)[0]
annotated = preds.plot()
annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8, 6))
plt.imshow(annotated_rgb)
plt.title('YOLOv8 Inference Result')
plt.axis('off')
plt.show()